# 06 — Q4: does a model fit on Diamond hold at Challenger?

Q1 asks whether the 10-minute state predicts the winner. Q4 asks the harder question:
**does the same relationship hold at a different skill level?**

This is the project's "decision making on new examples" result. The Challenger tier is never
trained on — it is held out entirely, as a genuinely unseen population rather than a random
slice of the same one.

> **STATUS: placeholder model.** The estimator below is a depth-4 decision tree standing in
> for B's chosen model. When B settles on one, replace the `MODEL = ...` line and re-run.
> Nothing else in this notebook changes.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

from lol import load, transfer, viz

viz.set_style()
pd.set_option("display.width", 200)

games = pd.read_parquet(load.processed_dir() / "games.parquet")
games.tier.value_counts().to_frame("games")

,games
tier,
challenger,23876
diamond,9879


## The split

Diamond is split into train and held-out so the within-tier number is honest; without that
the transfer gap would be confounded by memorisation. Challenger is touched only at scoring
time. `tier` itself is never a feature — a model that could see which file a row came from
would learn the file rather than the game, and the result would mean nothing.

In [2]:
# ---- swap this line for B's model ----------------------------------------
MODEL = DecisionTreeClassifier(max_depth=4, random_state=42)
# --------------------------------------------------------------------------

report = transfer.transfer_report(MODEL, games, home_tier="diamond")
report.round(4)

,accuracy,precision,recall,f1,roc_auc,n
diamond (train),0.7349,0.7401,0.7225,0.7312,0.8085,7409
diamond (held out),0.7215,0.7288,0.7040,0.7162,0.7966,2470
away (transfer),0.7454,0.7513,0.7331,0.7421,0.8147,23876


In [3]:
print(transfer.diagnose(report))

Ranking holds while the operating point improves (ROC-AUC +0.018, accuracy +0.024) — a calibration shift, not a broken relationship. No action needed; the home threshold is if anything conservative away.


Two different failures are possible and they mean different things, which is why both
ROC-AUC and accuracy are reported:

* **discrimination** falling (ROC-AUC) means the features rank games differently at the new
  tier — the relationship itself differs;
* **calibration** drifting (accuracy moving while ROC-AUC holds) means the ranking still
  works but the decision threshold sits in the wrong place.

A caveat that belongs in the write-up either way: the two tiers are different populations of
different sizes, so these numbers are **not** a like-for-like contest. A higher score away
does not mean the model is "better at Challenger".

In [4]:
fig, ax = plt.subplots(figsize=(7.4, 3.8))
rows = ["diamond (held out)", "away (transfer)"]
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
width = 0.38
for i, (row, colour) in enumerate(zip(rows, (viz.SERIES_1, viz.SERIES_2))):
    xs = [j + (i - 0.5) * width for j in range(len(metrics))]
    ax.bar(xs, report.loc[row, metrics].values, width=width, color=colour,
           label=f"{row} (n={report.loc[row, 'n']:,})")
ax.set_xticks(range(len(metrics)))
ax.set_xticklabels([m.replace("roc_auc", "ROC-AUC").title() for m in metrics])
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.set_title("Diamond-trained model, scored on held-out Diamond and on Challenger")
ax.legend(frameon=False, loc="lower right", fontsize=9.5)
ax.grid(axis="x", visible=False)
plt.show()

<Figure size 814x418 with 1 Axes>

## Do the same features matter at both tiers?

Equal scores could still hide a model leaning on completely different evidence. Fitting the
same estimator separately on each tier and comparing what it weighted answers that.

In [5]:
shift = transfer.feature_importance_shift(MODEL, games)
shift.head(8).round(4) if shift is not None else "This estimator exposes no importances or coefficients."

,diamond,challenger,shift
total_gold_diff,0.9564,0.9699,0.0135
total_jungle_minions_diff,0.0069,0.0000,0.0069
dragons_diff,0.0352,0.0301,0.0051
kills_diff,0.0015,0.0000,0.0015
wards_placed_diff,0.0000,0.0000,0.0000
wards_destroyed_diff,0.0000,0.0000,0.0000
first_blood_diff,0.0000,0.0000,0.0000
assists_diff,0.0000,0.0000,0.0000


## What to write up

Fill this in once B's model is in place. The three things the report needs:

1. the transfer numbers and which of the two failure modes (if either) appeared;
2. whether the features carrying the model are the same at both skill levels;
3. an honest statement of what this does **not** show — one region, one patch window, two
   tiers that are not a like-for-like comparison, and an observational dataset in which a
   gold lead predicting a win is not evidence that chasing gold early causes wins.